# Data Preparation

The notebook will show all the options available for preparing the data before sending it to the model.

In [1]:
import numpy as np

from sckitflow.data import DataManagerConfig
from sckitflow.data.sim import get_dummy_adata
from sckitflow.dataset.toy_data import get_toy_dataset
from sklearn.decomposition import PCA

rng = np.random.default_rng(0)

## Preliminaries

### Dummy Datasets

`sckitflow` allows you to create different types of dummy datasets using `sklearn.datasets` and wraps it in `Anndata` format using the function `get_toy_dataset(name, **kwargs)` where `name` refers to the type of dataset to be created and `**kwargs` are the additional parameters specific to dataset type. 

The following code block shows all the possible dataset types for creating dummy data

In [2]:
blobs_data = get_toy_dataset("blobs").adata
checkerboard_data = get_toy_dataset("checkerboard").adata
circles_data = get_toy_dataset("circles").adata
moons_data = get_toy_dataset("moons").adata
s_curve_data = get_toy_dataset("s_curve").adata
swiss_roll_data = get_toy_dataset("swiss_roll").adata

Investigating the data, we can see that the anndata structure varies in terms of fields with each data type. However, all of them have `adata.uns['dataset_info]` which contains all the parameters used in dataset creation. The default parameters remain the same from `sklearn.datasets` default values.

In [3]:
print(blobs_data)
print(checkerboard_data)

AnnData object with n_obs × n_vars = 1000 × 2
    uns: 'dataset_info'
    obsm: 'Y'
    layers: None (.X)
AnnData object with n_obs × n_vars = 1000 × 2
    uns: 'dataset_info'
    obsm: 'row_cluster'
    varm: 'col_cluster'
    layers: None (.X)


In [4]:
blobs_data.uns["dataset_info"]

{'name': 'blobs',
 'random_state': 42,
 'No of centers': 3,
 'cluster_std': 1.0,
 'center_box': (-10.0, 10.0),
 'shuffle': True,
 'centers': {'0': array([-2.50919762,  9.01428613]),
  '1': array([4.63987884, 1.97316968]),
  '2': array([-6.87962719, -6.88010959])}}

### PCA Representation

Preprocessing such as PCA is **not** performed by `sckitflow` — the caller computes the representation with their tool of choice and stores it in `.obsm`. Here we use scikit-learn's `PCA` and later point `DataManager` at it via `target_sample_rep`. Fitted attributes such as `pca.components_`, `pca.explained_variance_`, `pca.mean_`, and `pca.n_components_` live on the estimator.

In [5]:
blobs_data = get_toy_dataset("blobs", n_features=200, centers=5).adata

pca = PCA(n_components=50)
blobs_data.obsm["X_pca"] = pca.fit_transform(blobs_data.X)

print(f"Original Shape: {blobs_data.X.shape}")
print(f"New Shape: {blobs_data.obsm['X_pca'].shape}")

Original Shape: (1000, 200)
New Shape: (1000, 50)


## DataManager

### Basic Initialization

Creating Dummy data with 5000 samples and 200 features

In [6]:
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
pca = PCA(n_components=50)
dummy_data.obsm["X_pca"] = pca.fit_transform(dummy_data.X)
print(dummy_data)
print(dummy_data.obsm["X_pca"].shape)

AnnData object with n_obs × n_vars = 5000 × 200
    uns: 'dataset_info'
    obsm: 'Y', 'X_pca'
    layers: None (.X)
(5000, 50)


Initializing `DataManager` and asking it what the model is sized from: `get_data_dimensionalities` reads only the shapes of the representations the schema names (and encodes one group to size the encodings), without loading any data.

In [7]:
dm = DataManagerConfig(target_sample_rep="X").build()
print(dm.get_data_dimensionalities(dummy_data))

DataDimensions(state_dim=200, feature_names=Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       ...
       '190', '191', '192', '193', '194', '195', '196', '197', '198', '199'],
      dtype='str', length=200), condition_reps_dims={}, condition_continuous_dims={}, response_reps_dims={}, response_continuous_dims={}, groups_reps_dims={}, source_lin_dim=200, source_quad_dim=None, target_lin_dim=200, target_quad_dim=None, source_state_dim=None)


Initializing DataManager with the PCA representation and compiling the data. A rep names where it lives: `"X"`, `"obsm.<key>"` or `"layers.<key>"`; a bare key is rejected.

In [8]:
dm1 = DataManagerConfig(target_sample_rep="obsm.X_pca").build()
print(dm1.get_data_dimensionalities(dummy_data))

DataDimensions(state_dim=50, feature_names=Index(['X_pca_0', 'X_pca_1', 'X_pca_2', 'X_pca_3', 'X_pca_4', 'X_pca_5',
       'X_pca_6', 'X_pca_7', 'X_pca_8', 'X_pca_9', 'X_pca_10', 'X_pca_11',
       'X_pca_12', 'X_pca_13', 'X_pca_14', 'X_pca_15', 'X_pca_16', 'X_pca_17',
       'X_pca_18', 'X_pca_19', 'X_pca_20', 'X_pca_21', 'X_pca_22', 'X_pca_23',
       'X_pca_24', 'X_pca_25', 'X_pca_26', 'X_pca_27', 'X_pca_28', 'X_pca_29',
       'X_pca_30', 'X_pca_31', 'X_pca_32', 'X_pca_33', 'X_pca_34', 'X_pca_35',
       'X_pca_36', 'X_pca_37', 'X_pca_38', 'X_pca_39', 'X_pca_40', 'X_pca_41',
       'X_pca_42', 'X_pca_43', 'X_pca_44', 'X_pca_45', 'X_pca_46', 'X_pca_47',
       'X_pca_48', 'X_pca_49'],
      dtype='str'), condition_reps_dims={}, condition_continuous_dims={}, response_reps_dims={}, response_continuous_dims={}, groups_reps_dims={}, source_lin_dim=200, source_quad_dim=None, target_lin_dim=200, target_quad_dim=None, source_state_dim=None)


### Advanced Initialization

The initialization of `Datamanager` can be classified into 5 groups -

* Group Data
* State Data
* Coupling Data
* Condition Data
* Response Data

#### Group Data

This group of parameters control how the oberservation metadata is turned into numerical encoding. The parameters are -

* `groups`: Collection of `.obs` identifiers used to define grouping.


* `groups_reps`: Dictionary mapping `.obs` identifiers to pre defined numerical encoding which are stored in `.uns`


* `groups_encoding`: This parameter is used if the encoding have not been created and stored in `.uns` and we want to create the encoding now. The input is a dictionary mapping the `groups` identifiers to an encoder. Each group column takes either `groups_reps` or `groups_encoding` — never both.


* __NOTE__ Encoders are instances from `sckitflow.data.group_encoders` (`from sckitflow.data import group_encoders`): `LabelEncoderConfig`, `OneHotEncoderConfig`, `IdentityTransformerConfig`, `Log1pTransformerConfig`, `AffineTransformerConfig`. These are frozen configs of plain scalars rather than callables, which is what lets a whole `DataManager` be serialized. The categorical ones list their categories explicitly, as in `OneHotEncoderConfig(categories=("control", "drugA"))`, so the encoding never depends on which data it is built on; unknown categories raise.

In [9]:
# Creating Dummy data
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata

# Creating dummy .obs columns
drug_labels = ["drugA", "control"]
ko_labels = ["koA", "control"]
dummy_data.obs["drug"] = rng.choice(drug_labels, size=dummy_data.n_obs)
dummy_data.obs["ko"] = rng.choice(ko_labels, size=dummy_data.n_obs)

In [10]:
from sckitflow.data import group_encoders

dm = DataManagerConfig(
    target_sample_rep="X",
    groups=["drug", "ko"],
    groups_encoding={
        "drug": group_encoders.OneHotEncoderConfig(categories=tuple(sorted(dummy_data.obs["drug"].unique()))),
        "ko": group_encoders.LabelEncoderConfig(classes=tuple(sorted(dummy_data.obs["ko"].unique()))),
    },
).build()
dm.get_data_dimensionalities(dummy_data).groups_reps_dims  # one-hot: 1 column per drug; label: 1

{'drug': 2, 'ko': 1}

The schema does not split the data itself -- it *describes* it. The grouping becomes visible when the
data is **streamed**: `get_eval_loader` walks each unique `groups` combination once and yields the batch
together with its `leaf`, the tuple of group values it came from. Here that is the 4 combinations
`(drugA, koA)`, `(control, koA)`, `(drugA, control)`, `(control, control)`.

In [11]:
eval_loader = dm.get_eval_loader(dummy_data)

print(f"{len(eval_loader)} groups, ordered as {eval_loader.group_cols}")
for _, leaf in eval_loader:
    print(leaf)

4 groups, ordered as ('drug', 'ko')
('control', 'control')
('control', 'koA')
('drugA', 'control')
('drugA', 'koA')


In [12]:
# One group's batch: `leaf` says which group it is, `target_state` holds that group's cells.
step_data, leaf = next(iter(dm.get_eval_loader(dummy_data)))
print(leaf, step_data["target_state"].shape)

('control', 'control') torch.Size([1287, 200])


We can use `DataManager.groups_data_schema` to look at our variables

In [13]:
print(dm.groups_data_schema.groups)
print(dm.groups_data_schema.groups_encoders)
print(dm.groups_data_schema.groups_reps)

('drug', 'ko')
{'drug': OneHotEncoderConfig(categories=('control', 'drugA')), 'ko': LabelEncoderConfig(classes=('control', 'koA'))}
{}


#### State Data

This is just the basic initialization. The only parameter here is `target_sample_rep` by which you can choose which representation you want to use as input. It names where the state lives, `"X"`, `"obsm.<key>"` or `"layers.<key>"`, and is required: nothing defaults to `.X`.

In [14]:
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
pca = PCA(n_components=50)
dummy_data.obsm["X_pca"] = pca.fit_transform(dummy_data.X)

dm = DataManagerConfig(target_sample_rep="obsm.X_pca").build()
print(dm.get_data_dimensionalities(dummy_data))

DataDimensions(state_dim=50, feature_names=Index(['X_pca_0', 'X_pca_1', 'X_pca_2', 'X_pca_3', 'X_pca_4', 'X_pca_5',
       'X_pca_6', 'X_pca_7', 'X_pca_8', 'X_pca_9', 'X_pca_10', 'X_pca_11',
       'X_pca_12', 'X_pca_13', 'X_pca_14', 'X_pca_15', 'X_pca_16', 'X_pca_17',
       'X_pca_18', 'X_pca_19', 'X_pca_20', 'X_pca_21', 'X_pca_22', 'X_pca_23',
       'X_pca_24', 'X_pca_25', 'X_pca_26', 'X_pca_27', 'X_pca_28', 'X_pca_29',
       'X_pca_30', 'X_pca_31', 'X_pca_32', 'X_pca_33', 'X_pca_34', 'X_pca_35',
       'X_pca_36', 'X_pca_37', 'X_pca_38', 'X_pca_39', 'X_pca_40', 'X_pca_41',
       'X_pca_42', 'X_pca_43', 'X_pca_44', 'X_pca_45', 'X_pca_46', 'X_pca_47',
       'X_pca_48', 'X_pca_49'],
      dtype='str'), condition_reps_dims={}, condition_continuous_dims={}, response_reps_dims={}, response_continuous_dims={}, groups_reps_dims={}, source_lin_dim=200, source_quad_dim=None, target_lin_dim=200, target_quad_dim=None, source_state_dim=None)


In [15]:
dm.state_data_schema.sample_rep

'X_pca'

#### Coupling Data

This group of parameters control how the source and the target data distribution is coupled. The parameters are-

1. `target_sample_rep`: To choose the sourse representation

2. `target_rep`: To choose the target representation

3. `n_shared_dims`: If for some reason, the source and target embedding is not in the same latent space, then we have the option to choose the first n dimesions from both embeddings to sort of create a latent space.

In [16]:
n_obs_pert = 5000
n_obs_ctrl = 200
adata = get_dummy_adata(n_obs_pert=n_obs_pert, n_obs_ctrl=n_obs_ctrl, rng=rng)

In [17]:
# Fused coupling: a shared 10-dim linear term, plus each side's remaining features as its own quadratic.
n_lin = 10
adata.obsm["X_lin"] = adata.obsm["X_tgt"][:, :n_lin]
adata.obsm["X_src_quad"] = adata.obsm["X_src"][:, n_lin:]
adata.obsm["X_tgt_quad"] = adata.obsm["X_tgt"][:, n_lin:]

dm = DataManagerConfig(
    target_sample_rep="obsm.X_tgt",  # the target state the model sees
    rep_lin="obsm.X_lin",  # shared linear term (was: the first `n_shared_dims` of each rep)
    source_rep_quad="obsm.X_src_quad",  # source-only quadratic
    target_rep_quad="obsm.X_tgt_quad",  # target-only quadratic
).build()
dm.get_data_dimensionalities(adata)

DataDimensions(state_dim=22, feature_names=Index(['X_tgt_0', 'X_tgt_1', 'X_tgt_2', 'X_tgt_3', 'X_tgt_4', 'X_tgt_5',
       'X_tgt_6', 'X_tgt_7', 'X_tgt_8', 'X_tgt_9', 'X_tgt_10', 'X_tgt_11',
       'X_tgt_12', 'X_tgt_13', 'X_tgt_14', 'X_tgt_15', 'X_tgt_16', 'X_tgt_17',
       'X_tgt_18', 'X_tgt_19', 'X_tgt_20', 'X_tgt_21'],
      dtype='str'), condition_reps_dims={}, condition_continuous_dims={}, response_reps_dims={}, response_continuous_dims={}, groups_reps_dims={}, source_lin_dim=10, source_quad_dim=None, target_lin_dim=10, target_quad_dim=12, source_state_dim=None)

In [18]:
print(dm.coupling_data_schema.rep_lin)
print(dm.coupling_data_schema.target_rep_quad)
print(dm.coupling_data_schema.source_rep_quad)

X_lin
X_tgt_quad
X_src_quad


#### Condition Data

In [19]:
# regenerate data with a categorical condition column for this section
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
dummy_data.obs["drug"] = rng.choice(["drugA", "control"], size=dummy_data.n_obs)
dummy_data

AnnData object with n_obs × n_vars = 5000 × 200
    obs: 'drug'
    uns: 'dataset_info'
    obsm: 'Y'
    layers: None (.X)

In [20]:
# a categorical condition is encoded through a `.uns` table keyed by its values
dummy_data.uns["drug"] = {v: rng.standard_normal(8) for v in dummy_data.obs["drug"].unique()}
dm = DataManagerConfig(
    target_sample_rep="X",
    categorical_conditions={"drug": ("drug",)},
    categorical_reps={"drug": "drug"},
    control_values_dict={"drug": "control"},
).build()
dm.get_data_dimensionalities(dummy_data).condition_reps_dims

{'drug': 8}

#### Response Data

## Streaming to the model

Configuring the schema is only half of it -- the model never sees an `AnnData`, it sees batches. Which observations are held out is the experiment's decision: it writes a split column beforehand (e.g. with [flowrun](https://github.com/selmanozleyen/flowrun)'s split step), and the `DataManager` declares it with `split_by`. `get_dataloaders` turns it into one streaming loader per split.

In [21]:
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
dummy_data.obs["drug"] = rng.choice(["drugA", "drugB", "drugC", "drugD", "control"], size=dummy_data.n_obs)
dummy_data.obs["ko"] = rng.choice(["koA", "koB"], size=dummy_data.n_obs)

# Categorical conditions are encoded through `.uns`, keyed by the condition value.
dummy_data.uns["drug"] = {v: rng.standard_normal((1, 8)) for v in dummy_data.obs["drug"].unique()}

# Hold out the whole (drugD, koB) combination. Controls stay in train: they are the shared source.
held_out = (dummy_data.obs["drug"] == "drugD") & (dummy_data.obs["ko"] == "koB")
dummy_data.obs["split"] = np.where(held_out, "test", "train")
dummy_data.obs["split"].value_counts()

split
train    4507
test      493
Name: count, dtype: int64

`get_dataloaders` returns `{split: loader}`. Controls are shared across splits: every split draws its sources from all of them.

In [22]:
dm = DataManagerConfig(
    target_sample_rep="X",
    groups=["ko"],
    categorical_conditions={"drug": ["drug"]},
    categorical_reps={"drug": "drug"},
    groups_encoding={"ko": group_encoders.LabelEncoderConfig(classes=tuple(sorted(dummy_data.obs["ko"].unique())))},
    control_values_dict={"drug": "control"},
    split_by="split",  # the column the splitter above wrote
).build()

loaders = dm.get_dataloaders(dummy_data, batch_size=128)
print({split: f"{len(loader)} batches" for split, loader in loaders.items()})

{'train': '39 batches', 'test': '39 batches'}


Each loader yields a ready `StepData`: the perturbed cells as `target_state`, their matched controls as `source_state`, and the group/condition encodings tiled to the batch.

In [23]:
step_data = next(iter(loaders["train"]))

print("target:", step_data["target_state"].shape)
print("source:", step_data["source_state"].shape)
print("conditions:", {k: tuple(v.shape) for k, v in (step_data["target_condition_data"] or {}).items()})
print("groups:", {k: tuple(v.shape) for k, v in (step_data["target_group_data"] or {}).items()})

target: torch.Size([128, 200])
source: torch.Size([128, 200])
conditions: {}
groups: {'drug': (1, 1, 8), 'ko': (1, 1, 1)}
